# B4 — Fine-tuning PEGASUS avec LoRA (GPU T4)
**Personne B** | Phase 2 : Pipeline & Modèle

> **Input :** `donnees_propres.json` ← Personne A  
> **Output :** `model_finetuned/` → pour Personne A (Pipeline Map-Reduce)

**Technique :** LoRA + quantization 8-bit → PEGASUS dans 14 GB VRAM

**Fix v2 :**
- ✅ `tokenizer` chargé depuis `google/pegasus-xsum` (pas depuis `model_finetuned/`) — corrige le `TypeError: not a string`
- ✅ `tokenizer.save_pretrained()` explicite après chaque sauvegarde du modèle
- ✅ Diagnostic du dossier `model_finetuned/` à la fin

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 0. Installation

In [2]:
import subprocess
subprocess.run([
    "pip", "install",
    "transformers", "rouge-score", "sentencepiece",
    "accelerate", "bitsandbytes", "peft", "-q"
], check=True)
print("✅ Dépendances installées")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
✅ Dépendances installées


## 1. Imports & Configuration

In [3]:
import os, gc, json, time, torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import get_peft_model, LoraConfig, TaskType
from rouge_score import rouge_scorer as rs

# ✅ FIX : MODEL_BASE = source du tokenizer (toujours HuggingFace)
MODEL_NAME = "google/pegasus-xsum"
MAX_INPUT  = 512
MAX_TARGET = 256

HYPERPARAMS = {
    "num_epochs":                  5,
    "batch_size":                  1,
    "learning_rate":               1e-4,
    "weight_decay":                0.001,
    "warmup_steps":                200,
    "gradient_accumulation_steps": 4,
    "output_dir":                  "model_finetuned"
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device     : {device}")
print(f"Modele     : {MODEL_NAME}")
print(f"Max input  : {MAX_INPUT} tokens")
print(f"Max target : {MAX_TARGET} tokens")
print(f"LR         : {HYPERPARAMS['learning_rate']}")

if device == "cpu":
    print()
    print("ATTENTION : pas de GPU detecte !")
    print("  -> Le fine-tuning sur CPU sera TRES lent (~8h pour 1 epoch)")
    print("  -> Recommande : utiliser Google Colab avec GPU T4 (gratuit)")
    print("  -> Lien : https://colab.research.google.com")

Device     : cuda
Modele     : google/pegasus-xsum
Max input  : 512 tokens
Max target : 256 tokens
LR         : 0.0001


## 2. Classe BookSumDataset

In [4]:
class BookSumDataset(Dataset):
    def __init__(self, donnees, tokenizer, max_length=512):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        # Aplatir : chaque chunk devient un exemple séparé avec le résumé du livre
        self.paires = []
        for item in donnees:
            for chunk in item['chunks']:
                self.paires.append({
                    'chunk':  chunk,
                    'resume': item['resume']
                })
        print(f"   Dataset : {len(donnees)} livres → {len(self.paires)} paires chunk-résumé")

    def __len__(self):
        return len(self.paires)

    def __getitem__(self, idx):
        exemple = self.paires[idx]
        texte   = exemple['chunk']   # ← un seul chunk, taille correcte
        resume  = exemple['resume']

        entree = self.tokenizer(
            texte,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        sortie = self.tokenizer(
            resume,
            max_length=MAX_TARGET,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids":      entree["input_ids"].squeeze(),
            "attention_mask": entree["attention_mask"].squeeze(),
            "labels":         sortie["input_ids"].squeeze()
        }

print("✅ Classe BookSumDataset définie")

✅ Classe BookSumDataset définie


## 3. Chargement des données & DataLoaders

In [5]:
# ── Adapter le chemin selon l'environnement ──────────────────────────
# Kaggle :
DATA_PATH = "/kaggle/input/datasets/nohaqorrych/dataset/donnees_propres.json"
# Colab  : DATA_PATH = "donnees_propres.json"
# Local  : DATA_PATH = "donnees_propres.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    donnees = json.load(f)

print(f"✅ {len(donnees)} exemples chargés")

# ✅ FIX TOKENIZER : toujours charger depuis HuggingFace, jamais depuis model_finetuned/
# Raison : PEFT/LoRA ne sauvegarde pas spiece.model dans model_finetuned/
# → PegasusTokenizer.from_pretrained('model_finetuned/') lève TypeError: not a string
print(f"Chargement du tokenizer depuis '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # ← toujours HuggingFace
print(f"✅ Tokenizer chargé | vocab_size = {tokenizer.vocab_size:,}")

split = int(len(donnees) * 0.8)
train_dataset = BookSumDataset(donnees[:split], tokenizer, MAX_INPUT)
val_dataset   = BookSumDataset(donnees[split:], tokenizer, MAX_INPUT)

def collate_fn(batch):
    input_ids      = torch.stack([b["input_ids"]      for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    labels         = torch.stack([b["labels"]         for b in batch])
    labels[labels == tokenizer.pad_token_id] = -100
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_loader = DataLoader(
    train_dataset,
    batch_size=HYPERPARAMS["batch_size"],
    shuffle=True,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_size=HYPERPARAMS["batch_size"],
    shuffle=False,
    collate_fn=collate_fn
)

print(f"   Train : {len(train_dataset)} exemples | {len(train_loader)} batches")
print(f"   Val   : {len(val_dataset)} exemples | {len(val_loader)} batches")

✅ 2000 exemples chargés
Chargement du tokenizer depuis 'google/pegasus-xsum'...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

✅ Tokenizer chargé | vocab_size = 96,103
   Dataset : 1600 livres → 17435 paires chunk-résumé
   Dataset : 400 livres → 3937 paires chunk-résumé
   Train : 17435 exemples | 17435 batches
   Val   : 3937 exemples | 3937 batches


## 4. Chargement PEGASUS en 8-bit + LoRA

In [6]:
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    print("Chargement de PEGASUS en 8-bit...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    print("CPU détecté → chargement sans quantization...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32
    )

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    bias="none"
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Trainable parameters: {trainable_params:,}")
if trainable_params == 0:
    raise RuntimeError(
        "Aucun paramètre entraînable trouvé pour LoRA. "
        "Vérifiez target_modules et l'architecture du modèle."
    )

model.to(device)
model.config.use_cache = False
model.gradient_checkpointing_disable()
print("\n✅ PEGASUS + LoRA chargé (checkpointing désactivé)")

if torch.cuda.is_available():
    vram_used  = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM utilisée : {vram_used:.1f} GB / {vram_total:.1f} GB")
    print(f"   VRAM libre    : {vram_total - vram_used:.1f} GB pour l'entraînement")

Chargement de PEGASUS en 8-bit...


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

trainable params: 6,291,456 || all params: 772,858,880 || trainable%: 0.8140
   Trainable parameters: 6,291,456

✅ PEGASUS + LoRA chargé (checkpointing désactivé)
   VRAM utilisée : 1.7 GB / 15.6 GB
   VRAM libre    : 14.0 GB pour l'entraînement


## 5. Optimizer & Scheduler

In [7]:
trainable_param_list = [p for p in model.parameters() if p.requires_grad]
if not trainable_param_list:
    raise RuntimeError(
        "Aucun paramètre entraînable trouvé pour l'optimiseur. "
        "Vérifiez la configuration LoRA et les target_modules."
    )

optimizer = AdamW(
    trainable_param_list,
    lr=HYPERPARAMS["learning_rate"],
    weight_decay=HYPERPARAMS["weight_decay"]
)

total_steps = len(train_loader) * HYPERPARAMS["num_epochs"]
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=HYPERPARAMS["warmup_steps"],
    num_training_steps=total_steps
)

print(f"✅ Optimizer : AdamW (lr={HYPERPARAMS['learning_rate']})")
print(f"✅ Scheduler : Linear warmup ({HYPERPARAMS['warmup_steps']} steps)")
print(f"   Total steps : {total_steps}")

✅ Optimizer : AdamW (lr=0.0001)
✅ Scheduler : Linear warmup (200 steps)
   Total steps : 87175


## 6. Boucle de Fine-tuning avec Early Stopping

In [ ]:
os.makedirs(HYPERPARAMS["output_dir"], exist_ok=True)
scorer      = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
historique  = []
meilleur_r1 = 0.0

# ✅ Early Stopping
PATIENCE     = 2
ES_MIN_DELTA = 0.005
es_counter   = 0
early_stop   = False
grad_acc     = HYPERPARAMS["gradient_accumulation_steps"]

for epoch in range(1, HYPERPARAMS["num_epochs"] + 1):

    if early_stop:
        print(f"\n⛔ Early stopping déclenché à l'epoch {epoch}")
        print(f"   Pas d'amélioration depuis {PATIENCE} epochs (delta < {ES_MIN_DELTA})")
        break

    print(f"\n{'='*60}")
    print(f" EPOCH {epoch}/{HYPERPARAMS['num_epochs']}  [patience: {es_counter}/{PATIENCE}]")
    print(f"{'='*60}")

    # ── TRAIN ────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    t0         = time.time()
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader, 1):
        try:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss / grad_acc

            if not loss.requires_grad:
                raise RuntimeError("loss.requires_grad is False")

            loss.backward()
            total_loss += outputs.loss.item()

            if step % grad_acc == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            if step % 10 == 0:
                print(f"   Step {step:>3}/{len(train_loader)} | Loss: {total_loss/step:.4f}")

        except RuntimeError as e:
            print(f"   ⚠️ Erreur step {step} : {e}")
            optimizer.zero_grad()
            continue

        finally:
            del input_ids, attention_mask, labels, outputs
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    train_loss = total_loss / len(train_loader)
    train_time = round(time.time() - t0, 1)

    # ── VALIDATION ───────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    rouge1_list, rouge2_list, rougeL_list = [], [], []

    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            try:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device)

                outputs   = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                val_loss += outputs.loss.item()

                if i < 3:
                    gen_ids = model.generate(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=200,
                        num_beams=4,
                        length_penalty=1.5,
                        no_repeat_ngram_size=3,
                        early_stopping=True,
                    )
                    preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
                    refs  = tokenizer.batch_decode(
                        torch.where(
                            labels == -100,
                            torch.tensor(tokenizer.pad_token_id, device=labels.device),
                            labels
                        ),
                        skip_special_tokens=True
                    )
                    for pred, ref in zip(preds, refs):
                        if ref.strip():
                            s = scorer.score(ref, pred)
                            rouge1_list.append(s["rouge1"].fmeasure)
                            rouge2_list.append(s["rouge2"].fmeasure)
                            rougeL_list.append(s["rougeL"].fmeasure)

            except RuntimeError as e:
                print(f"   ⚠️ Erreur val step {i} : {e}")
                continue

            finally:
                del input_ids, attention_mask, labels, outputs
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    val_loss_avg = val_loss / len(val_loader)
    r1 = round(sum(rouge1_list) / len(rouge1_list), 4) if rouge1_list else 0.0
    r2 = round(sum(rouge2_list) / len(rouge2_list), 4) if rouge2_list else 0.0
    rl = round(sum(rougeL_list) / len(rougeL_list), 4) if rougeL_list else 0.0

    print(f"\n  📉 Train Loss : {train_loss:.4f}  ({train_time}s)")
    print(f"  📉 Val Loss   : {val_loss_avg:.4f}")
    print(f"  📊 ROUGE-1    : {r1}")
    print(f"  📊 ROUGE-2    : {r2}")
    print(f"  📊 ROUGE-L    : {rl}")

    historique.append({
        "epoch":      epoch,
        "train_loss": round(train_loss, 4),
        "val_loss":   round(val_loss_avg, 4),
        "rouge1":     r1,
        "rouge2":     r2,
        "rougeL":     rl
    })

    # ✅ Sauvegarde meilleur modèle + tokenizer
    if r1 > meilleur_r1 + ES_MIN_DELTA:
        meilleur_r1 = r1
        es_counter  = 0
        # Sauvegarder les poids LoRA
        model.save_pretrained(HYPERPARAMS["output_dir"])
        # ✅ FIX : sauvegarder explicitement le tokenizer (spiece.model inclus)
        # Sans cette ligne, model_finetuned/ ne contiendra pas spiece.model
        # et PegasusTokenizer.from_pretrained('model_finetuned/') plantera
        tokenizer.save_pretrained(HYPERPARAMS["output_dir"])
        print(f"  💾 Meilleur modèle sauvegardé (ROUGE-1={r1})")
        print(f"  💾 Tokenizer (spiece.model) sauvegardé dans {HYPERPARAMS['output_dir']}/")
    else:
        es_counter += 1
        print(f"  ⚠️  Pas d'amélioration ({es_counter}/{PATIENCE})")
        if es_counter >= PATIENCE:
            early_stop = True
            print(f"  🔴 Early stopping activé — meilleur ROUGE-1 = {meilleur_r1}")

print("\n✅ Fine-tuning terminé !")
print(f"   Meilleur ROUGE-1  : {meilleur_r1}")
print(f"   Epochs effectuées : {len(historique)}/{HYPERPARAMS['num_epochs']}")
print(f"   Early stop        : {'Oui' if early_stop else 'Non'}")


 EPOCH 1/5  [patience: 0/2]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


   Step  10/17435 | Loss: 4.9176
   Step  20/17435 | Loss: 4.8633
   Step  30/17435 | Loss: 4.8790
   Step  40/17435 | Loss: 4.7118
   Step  50/17435 | Loss: 4.6063
   Step  60/17435 | Loss: 4.6106
   Step  70/17435 | Loss: 4.6862
   Step  80/17435 | Loss: 4.6448
   Step  90/17435 | Loss: 4.6485
   Step 100/17435 | Loss: 4.6651
   Step 110/17435 | Loss: 4.7003
   Step 120/17435 | Loss: 4.6806
   Step 130/17435 | Loss: 4.6757
   Step 140/17435 | Loss: 4.6615
   Step 150/17435 | Loss: 4.6435
   Step 160/17435 | Loss: 4.6397
   Step 170/17435 | Loss: 4.6574
   Step 180/17435 | Loss: 4.6457
   Step 190/17435 | Loss: 4.6262
   Step 200/17435 | Loss: 4.6056
   Step 210/17435 | Loss: 4.6136
   Step 220/17435 | Loss: 4.5981
   Step 230/17435 | Loss: 4.5825
   Step 240/17435 | Loss: 4.5923
   Step 250/17435 | Loss: 4.5823
   Step 260/17435 | Loss: 4.5706
   Step 270/17435 | Loss: 4.5615
   Step 280/17435 | Loss: 4.5692
   Step 290/17435 | Loss: 4.5737
   Step 300/17435 | Loss: 4.5605
   Step 31

In [ ]:
!nvidia-smi

## 7. Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

epochs_list  = [h["epoch"]      for h in historique]
train_losses = [h["train_loss"] for h in historique]
val_losses   = [h["val_loss"]   for h in historique]
r1_vals      = [h["rouge1"]     for h in historique]
r2_vals      = [h["rouge2"]     for h in historique]
rl_vals      = [h["rougeL"]     for h in historique]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_list, train_losses, "o-", color="steelblue", label="Train Loss", linewidth=2)
axes[0].plot(epochs_list, val_losses,   "o-", color="coral",     label="Val Loss",   linewidth=2)
axes[0].set_title("Courbe de Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_xticks(epochs_list)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_list, r1_vals, "o-", color="#00d4aa", label="ROUGE-1", linewidth=2)
axes[1].plot(epochs_list, r2_vals, "o-", color="#f87171", label="ROUGE-2", linewidth=2)
axes[1].plot(epochs_list, rl_vals, "o-", color="#a78bfa", label="ROUGE-L", linewidth=2)
axes[1].set_title("Scores ROUGE (validation)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_xticks(epochs_list)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("B4 — Fine-tuning PEGASUS + LoRA sur BookSum", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("courbes_finetuning.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ courbes_finetuning.png sauvegardé")

## 8. Export rapport final

In [ ]:
meilleur = max(historique, key=lambda h: h["rouge1"])

rapport_final = {
    "modele":           MODEL_NAME,
    "technique":        "LoRA + 8-bit quantization",
    "output_dir":       HYPERPARAMS["output_dir"],
    "hyperparams":      HYPERPARAMS,
    "meilleure_epoch":  meilleur["epoch"],
    "early_stopping":   early_stop,
    "meilleurs_scores": {
        "rouge1": meilleur["rouge1"],
        "rouge2": meilleur["rouge2"],
        "rougeL": meilleur["rougeL"]
    },
    "historique": historique,
    "message_pour_A4": (
        f"Modèle LoRA disponible dans '{HYPERPARAMS['output_dir']}'. "
        "Charger avec : model = PeftModel.from_pretrained(base_model, 'model_finetuned'). "
        "IMPORTANT : charger le tokenizer depuis 'google/pegasus-xsum', pas depuis 'model_finetuned/'."
    )
}

with open("rapport_finetuning.json", "w", encoding="utf-8") as f:
    json.dump(rapport_final, f, indent=2, ensure_ascii=False)

print("=" * 55)
print(" RÉSULTATS FINAUX — B4 Fine-tuning PEGASUS + LoRA")
print("=" * 55)
print(f"  Meilleure epoch : {meilleur['epoch']}")
print(f"  ROUGE-1         : {meilleur['rouge1']}")
print(f"  ROUGE-2         : {meilleur['rouge2']}")
print(f"  ROUGE-L         : {meilleur['rougeL']}")
print(f"  Early stop      : {'Oui' if early_stop else 'Non'}")
print()
print(f"  💾 Modèle sauvegardé dans : {HYPERPARAMS['output_dir']}/")
print(f"  📄 rapport_finetuning.json sauvegardé")
print()
print("  ✅ Personne A peut démarrer A4 (Pipeline Map-Reduce)")

## 9. Diagnostic — Vérification des fichiers sauvegardés

In [ ]:
# ✅ Vérifier que tous les fichiers nécessaires sont présents
import os

dossier = HYPERPARAMS["output_dir"]
print(f"Contenu de '{dossier}/' :")
print("-" * 50)

fichiers_attendus = [
    ("adapter_config.json",        "Config LoRA"),
    ("adapter_model.safetensors",  "Poids LoRA"),
    ("tokenizer_config.json",      "Config tokenizer"),
    ("spiece.model",               "SentencePiece model ← OBLIGATOIRE pour PEGASUS"),
    ("special_tokens_map.json",    "Tokens spéciaux"),
]

if os.path.exists(dossier):
    fichiers_presents = set(os.listdir(dossier))
    for fichier, description in fichiers_attendus:
        present = "✅" if fichier in fichiers_presents else "❌ MANQUANT"
        taille  = ""
        if fichier in fichiers_presents:
            taille = f"({os.path.getsize(os.path.join(dossier, fichier)) / 1024:.0f} KB)"
        print(f"  {present}  {fichier:<40} {taille}  {description}")

    print()
    autres = fichiers_presents - {f for f, _ in fichiers_attendus}
    for f in sorted(autres):
        taille = os.path.getsize(os.path.join(dossier, f)) / 1024
        print(f"  ℹ️   {f:<40} ({taille:.0f} KB)")
else:
    print(f"❌ Dossier '{dossier}/' introuvable !")

print()
# Test de chargement du tokenizer
print("Test de rechargement du tokenizer...")
try:
    from transformers import AutoTokenizer
    tok_test = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"✅ Tokenizer rechargeable depuis '{MODEL_NAME}'")
    print(f"   vocab_size = {tok_test.vocab_size:,}")
except Exception as e:
    print(f"❌ Erreur : {e}")

In [ ]:
tokenizer.save_pretrained("model_finetuned")

In [ ]:
model.save_pretrained("model_finetuned")

## ✅ Résumé B4

| Élément | Valeur |
|---|---|
| Modèle | `google/pegasus-xsum` |
| Technique | LoRA (r=16) + quantization 8-bit |
| Paramètres entraînés | ~6M / 568M (< 1%) |
| Input max | 256 tokens |
| Output | `model_finetuned/` + `rapport_finetuning.json` |

**👉 Personne A charge le modèle dans A4 avec :**
```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

MODEL_BASE = "google/pegasus-xsum"

# ✅ CORRECT : tokenizer depuis HuggingFace (pas depuis model_finetuned/)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_BASE)

# Charger modèle de base + poids LoRA
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_BASE)
model      = PeftModel.from_pretrained(base_model, 'model_finetuned')
model      = model.merge_and_unload()  # fusion LoRA → modèle dense
model.eval()

# ❌ NE PAS FAIRE :
# tokenizer = PegasusTokenizer.from_pretrained('model_finetuned/')  
# → TypeError: not a string  (spiece.model manque dans model_finetuned/)
```